# Basic Docker & Kubernetes Hands-On

**Morning session · 9:40 – 10:50 AM** · [website version](https://training.nrp-nautilus.io/pearc26/2_kubernetes.html) — run cells with **Shift+Enter**.


## ⚙️ Setup — run this first

Set your short username once; every command below uses `$NRP_USER`. The cell
also renders every manifest into **`my-yamls/`** with `<username>` already
filled in — wherever the website says *"replace `<username>`"*, it's already
done for you here.

> Terminal steps below don't share this variable — run the same
> `export NRP_USER=...` line in any terminal you open.
> Re-running this cell re-renders `my-yamls/` (overwriting any edits you made there).


In [ ]:
export NRP_USER=changeme   # ✏️ EDIT to your short name, then Shift+Enter
cd ~/pearc26/workspace
if [ "$NRP_USER" = changeme ]; then echo "⚠️  Edit NRP_USER above first, then re-run"; else
  mkdir -p my-yamls
  for f in yamls/*; do sed "s/<username>/$NRP_USER/g" "$f" > "my-yamls/$(basename "$f")"; done
  echo "✅ my-yamls/ rendered for $NRP_USER"
fi


**Morning session · 9:40 – 10:50 AM**

This episode is the core Kubernetes hands-on: scheduling pods and jobs, persistent storage, multi-container pods, ConfigMaps and Secrets, Deployments, exposing an HTTPS service, steering pods with taints/tolerations and node affinity, and launching a GPU pod.

**Conventions.** Hands-on examples use the **`nrp-training-k8s`** namespace. In any YAML or command, replace **`<username>`** with a short version of your name or username to avoid collisions with other participants. Manifests live in the workspace `yamls/` folder.

> 📘 **Docs:** [Kubernetes basics](https://nrp.ai/documentation/userdocs/tutorial/basic/) · [GPU pods](https://nrp.ai/documentation/userdocs/running/gpu-pods/) · [Run jobs](https://nrp.ai/documentation/userdocs/running/jobs/) · [Storage](https://nrp.ai/documentation/userdocs/storage/intro/) · [Live resources](https://nrp.ai/viz/resources/)


## kubectl flags you'll reach for constantly

| Flag | Purpose |
|---|---|
| `-n <namespace>` | Target a specific namespace. |
| `-l key=value` | Filter resources by label. |
| `-w` / `--watch` | Stream live updates instead of a one-shot list. Ctrl-C to stop. |
| `-o wide` | Add columns: node, pod IP, container image, etc. |
| `-o yaml` / `-o json` | Print the full resource manifest. |
| `-o jsonpath='{...}'` | Extract one field. |
| `--show-labels` | Append a column with every label a resource carries. |
| `--previous` (on `kubectl logs`) | Logs from the *previous* container instance — essential for crashloops. |


## Hands-on: a simple pod

Open `yamls/test-pod.yaml` and replace `<username>` in `metadata.name`:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: test-pod-<username>
  namespace: nrp-training-k8s
spec:
  containers:
  - name: mypod
    image: ubuntu:22.04
    command: ["sh", "-c", "echo 'Hello from NRP!' && sleep 3600"]
    resources:
      limits:  { memory: 100Mi, cpu: 100m }
      requests: { memory: 100Mi, cpu: 100m }
```


Notice `requests` and `limits` are identical — the Gatekeeper-safe default from Episode 1.

Launch and inspect:


In [ ]:
kubectl apply -n nrp-training-k8s -f my-yamls/test-pod.yaml
kubectl get pods -n nrp-training-k8s
kubectl logs test-pod-$NRP_USER -n nrp-training-k8s


<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
pod/test-pod-&lt;username&gt; created

NAME                  READY   STATUS    RESTARTS   AGE
test-pod-&lt;username&gt;   1/1     Running   0          12s

Hello from NRP!
</pre>
</details>


Run a command inside it, then open an interactive shell (Ctrl-D to exit):


In [ ]:
kubectl exec test-pod-$NRP_USER -n nrp-training-k8s -- echo 'Command executed successfully'


**🖥️ Terminal step** — interactive or long-running: use a JupyterLab terminal (**File → New → Terminal**), not this notebook. Ctrl-C / Ctrl-D to exit.

```bash
kubectl exec -it test-pod-$NRP_USER -n nrp-training-k8s -- /bin/bash
```


**💡 The debugging trio**

When something doesn't behave the way you expect:


In [ ]:
kubectl describe pod test-pod-$NRP_USER -n nrp-training-k8s          # status + last events
kubectl get events -n nrp-training-k8s --sort-by=.metadata.creationTimestamp | tail -20
kubectl logs test-pod-$NRP_USER -n nrp-training-k8s --previous        # logs from the prior crash


`describe` shows scheduling decisions and container state; `get events` shows the namespace timeline; `--previous` is essential for crashlooping pods.


Clean up:


In [ ]:
kubectl delete pod test-pod-$NRP_USER -n nrp-training-k8s


## Hands-on: persistent storage with a PVC

Pods are ephemeral — anything written to the container filesystem disappears when the pod terminates. **PersistentVolumeClaims** ask Kubernetes for long-lived storage you can mount into pods. On NRP we typically use `rook-ceph-block-east` for general-purpose `ReadWriteOnce` block storage.

Open `yamls/pvc.yaml` — it contains a 1 GiB PVC and a writer pod that mounts it at `/data`. Replace `<username>` in both names, then:


In [ ]:
kubectl apply -n nrp-training-k8s -f my-yamls/pvc.yaml
kubectl get pvc -n nrp-training-k8s
kubectl get pod pvc-pod-$NRP_USER -n nrp-training-k8s


<details>
<summary>Expected output (Ceph provisioning takes ~30–60s on first claim)</summary>

<pre style="line-height:1.45">
NAME             STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS           AGE
pvc-&lt;username&gt;   Bound    pvc-99a63070-eb3d-490a-82fd-4e5811e4a5df   1Gi        RWO            rook-ceph-block-east   45s

NAME                 READY   STATUS    RESTARTS   AGE
pvc-pod-&lt;username&gt;   1/1     Running   0          47s
</pre>
</details>


Prove the data survives pod deletion — delete only the pod, re-apply, and read the file back:


In [ ]:
kubectl exec pvc-pod-$NRP_USER -n nrp-training-k8s -- cat /data/log.txt
kubectl delete pod pvc-pod-$NRP_USER -n nrp-training-k8s
kubectl apply -n nrp-training-k8s -f my-yamls/pvc.yaml
kubectl exec pvc-pod-$NRP_USER -n nrp-training-k8s -- cat /data/log.txt   # previous line still there


**Don't delete the PVC yet** — the next section reuses it.


## Hands-on: multi-container pod (sidecar pattern)

A pod can hold more than one container — they share the network namespace (same `localhost`) and any volumes mounted into both. This is the classic **sidecar** pattern: a main container plus a supporting one (log shipping, file syncing, format conversion).

`yamls/multicontainer.yaml` defines a pod whose **writer** container appends a tick line to a shared file every 5 seconds while a **reader** container tails the same file. It reuses `pvc-<username>` from the previous section — first delete the writer pod so the RWO volume detaches:


In [ ]:
kubectl delete pod pvc-pod-$NRP_USER -n nrp-training-k8s --ignore-not-found
kubectl apply -n nrp-training-k8s -f my-yamls/multicontainer.yaml
kubectl get pod sidecar-$NRP_USER -n nrp-training-k8s


Read each container's log stream separately with `-c`:


In [ ]:
kubectl logs sidecar-$NRP_USER -c writer -n nrp-training-k8s --tail=5
kubectl logs sidecar-$NRP_USER -c reader -n nrp-training-k8s --tail=5


<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
# writer:
writer-tick 1 04:55:58
writer-tick 2 04:56:04

# reader (tailing the shared file from a different process):
reader started, tailing /shared/data.log
writer-tick 1 04:55:58
writer-tick 2 04:56:04
</pre>
</details>


Every line the writer appends shows up in the reader's stream — both containers see the same volume. Clean up (this also releases the PVC):


In [ ]:
kubectl delete -n nrp-training-k8s -f my-yamls/multicontainer.yaml
kubectl delete -n nrp-training-k8s -f my-yamls/pvc.yaml


## Hands-on: ConfigMap, Secret, and env vars

Hard-coding paths, hostnames, or API tokens into images is a recipe for pain. Kubernetes gives you two purpose-built objects:

- **ConfigMap** — non-sensitive key/value config, stored as plain text.
- **Secret** — sensitive values (tokens, passwords, TLS keys), stored base64-encoded with separate RBAC.

`yamls/configmap-secret.yaml` ships a ConfigMap, a Secret, and a Pod that pulls ConfigMap keys in bulk via `envFrom` and the Secret via `secretKeyRef`. Replace `<username>` in all names and apply:


In [ ]:
kubectl apply -n nrp-training-k8s -f my-yamls/configmap-secret.yaml
kubectl logs env-pod-$NRP_USER -n nrp-training-k8s


<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
GREETING=Hello from NRP
SERVER_PORT=8080
API_TOKEN starts with: tutorial…
</pre>
</details>


Look inside each object:


In [ ]:
kubectl get configmap app-config-$NRP_USER -n nrp-training-k8s -o yaml | grep -A2 '^data:'
kubectl get secret    app-secret-$NRP_USER -n nrp-training-k8s -o jsonpath='{.data.API_TOKEN}' | base64 -d ; echo


Base64 is **storage format, not encryption** — anyone who can `get secret` in your namespace can read it. Clean up:


In [ ]:
kubectl delete -n nrp-training-k8s -f my-yamls/configmap-secret.yaml


## Hands-on: Deployment

A **Deployment** keeps a set of identical pods running: it restarts them when they fail and rolls out new versions without downtime. Open `yamls/deployment.yaml`, replace `<username>`, apply:


In [ ]:
kubectl apply -n nrp-training-k8s -f my-yamls/deployment.yaml
kubectl get deploy,rs,pod -n nrp-training-k8s -l app=hello-deploy-$NRP_USER


Try the basic operations:


In [ ]:
# scale to 4 replicas
kubectl scale deployment hello-deploy-$NRP_USER -n nrp-training-k8s --replicas=4

# delete one pod and watch the Deployment immediately recreate it
VICTIM=$(kubectl get pod -n nrp-training-k8s -l app=hello-deploy-$NRP_USER -o jsonpath='{.items[0].metadata.name}')
kubectl delete pod "$VICTIM" -n nrp-training-k8s
kubectl get pods -n nrp-training-k8s -l app=hello-deploy-$NRP_USER   # still 4

# rolling update to a different image
kubectl set image deployment/hello-deploy-$NRP_USER -n nrp-training-k8s hello=nginx:alpine
kubectl rollout status deployment/hello-deploy-$NRP_USER -n nrp-training-k8s


### Working with running pods: cp, port-forward, patch

Pick one pod from the Deployment:


In [ ]:
POD=$(kubectl get pod -n nrp-training-k8s -l app=hello-deploy-$NRP_USER -o jsonpath='{.items[0].metadata.name}')


Copy files in and out:


In [ ]:
echo "training data v1" > /tmp/dataset.txt
kubectl cp /tmp/dataset.txt nrp-training-k8s/"$POD":/tmp/dataset.txt
kubectl exec "$POD" -n nrp-training-k8s -- cat /tmp/dataset.txt


Tunnel a pod port to your terminal (foreground; use a second terminal for `curl`):

**🖥️ Terminal step** — interactive or long-running: use a JupyterLab terminal (**File → New → Terminal**), not this notebook. Ctrl-C / Ctrl-D to exit.

```bash
kubectl port-forward "$POD" -n nrp-training-k8s 8080:80
```


In [ ]:
# second terminal:
curl -s -o /dev/null -w "HTTP %{http_code}\n" http://localhost:8080


Patch a single field without re-applying YAML:


In [ ]:
kubectl patch deployment hello-deploy-$NRP_USER -n nrp-training-k8s -p '{"spec":{"replicas":2}}'


Clean up:


In [ ]:
kubectl delete -n nrp-training-k8s -f my-yamls/deployment.yaml


## Hands-on: batch Job

A **Job** runs pods until a target number complete successfully. Open `yamls/job.yaml`, replace `<username>`, apply, and watch π get computed:


In [ ]:
kubectl apply -n nrp-training-k8s -f my-yamls/job.yaml
kubectl get jobs -n nrp-training-k8s
kubectl logs -n nrp-training-k8s -l job-name=pi-$NRP_USER


<details>
<summary>Expected output (after 50–120s of CPU work)</summary>

<pre style="line-height:1.45">
3.14159265358979323846264338327950288419716939937510582097494459230781640628620…

NAME             STATUS     COMPLETIONS   DURATION   AGE
pi-&lt;username&gt;    Complete   1/1           53s        57s
</pre>
</details>


The Job auto-deletes 10 minutes after completion (`ttlSecondsAfterFinished: 600`).


## Hands-on: exposing a service over HTTPS

To expose an HTTP application publicly you need three objects: a **Deployment** (runs the pods), a **Service** (stable in-cluster name), and an **Ingress** on the `haproxy` class that routes a public hostname to the Service. NRP runs HAProxy as the ingress controller, and Cert Manager issues a free Let's Encrypt TLS certificate automatically for any `*.nrp-nautilus.io` hostname.

Open `yamls/ingress-demo.yaml` and replace **every** `<username>` (the hostname `hello-<username>.nrp-nautilus.io` must be globally unique):


In [ ]:
kubectl apply -n nrp-training-k8s -f my-yamls/ingress-demo.yaml
kubectl get deploy,svc,ingress -n nrp-training-k8s -l k8s-app=hello-web-$NRP_USER


Wait ~60 seconds for HAProxy and the certificate, then:


In [ ]:
curl -sI https://hello-$NRP_USER.nrp-nautilus.io | head -5


<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
HTTP/2 200
server: nginx/1.29.1
content-type: text/plain
</pre>
</details>


Open the URL in your browser and reload a few times — the `Server name` line cycles between the two replicas. Clean up (this releases the public hostname):


In [ ]:
kubectl delete -n nrp-training-k8s -f my-yamls/ingress-demo.yaml


## Scheduling: labels, affinity, taints, and tolerations

NRP is a heterogeneous shared cluster — 500+ nodes, many GPU SKUs, and pools reserved for specific projects. Scheduling primitives are how you say "*put my pod **here**, not **there***":

| Primitive | Lives on | Asks the question |
|---|---|---|
| **Node label** | Node | "What is this node? (GPU type, region, owner…)" |
| **`nodeSelector` / `nodeAffinity`** | Pod | "Which nodes am I willing to land on?" |
| **Taint** | Node | "Who is allowed to land here?" |
| **Toleration** | Pod | "I have permission to land on those tainted nodes." |

Labels + affinity are an **attraction**; taints + tolerations are a **repulsion**. You usually need **both**: a toleration to be allowed onto a reserved node, plus an affinity rule so the scheduler actually picks it.


### The PEARC26 reserved GPU pool

For the tutorial, NRP has a pool of NVIDIA A10 GPU nodes reserved:

- **Label** `nrp-training=true` — marks the tutorial nodes.
- **Taint** `nautilus.io/reservation=nrp:NoSchedule` — keeps other workloads off them.

Explore the pool:


In [ ]:
kubectl get nodes -l nrp-training=true -L nvidia.com/gpu.product
kubectl get nodes -l nrp-training=true \
  -o jsonpath='{range .items[*]}{.metadata.name}{"\t"}{.spec.taints}{"\n"}{end}'


<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
NAME                         STATUS   ROLES    AGE      VERSION    GPU.PRODUCT
hcc-nrp-shor-c5825.unl.edu   Ready    &lt;none&gt;   3y300d   v1.33.12   NVIDIA-A10
hcc-nrp-shor-c5905.unl.edu   Ready    &lt;none&gt;   3y300d   v1.33.8    NVIDIA-A10
…

hcc-nrp-shor-c5825.unl.edu	[{&quot;effect&quot;:&quot;NoSchedule&quot;,&quot;key&quot;:&quot;nautilus.io/reservation&quot;,&quot;value&quot;:&quot;nrp&quot;}, …]
</pre>
</details>


To land on the pool, a pod spec needs both blocks — this pattern appears in every GPU manifest in the workspace:

```yaml
spec:
  tolerations:
  - key: nautilus.io/reservation
    operator: Equal
    value: nrp
    effect: NoSchedule
  affinity:
    nodeAffinity:
      preferredDuringSchedulingIgnoredDuringExecution:
      - weight: 100
        preference:
          matchExpressions:
          - key: nrp-training
            operator: In
            values: ["true"]
```


`preferred…` is a soft hint — the scheduler picks a reserved node if one is free but won't strand your pod if all are busy. The `required…` variant blocks scheduling until a matching node frees up. Beyond this tutorial the same pattern targets specific GPU models (`nvidia.com/gpu.product=NVIDIA-A100-PCIE-40GB`), CUDA versions (`nvidia.com/cuda.runtime.major=12`), or regions (`topology.kubernetes.io/region=us-west`).


## Hands-on: your first GPU pod

`yamls/gpu-pod.yaml` requests one GPU via resource limits:

```yaml
    resources:
      limits:
        nvidia.com/gpu: 1
      requests:
        nvidia.com/gpu: 1
```


Resource keys by hardware type:

- **NVIDIA GPUs (generic):** `nvidia.com/gpu: <count>`
- **Qualcomm Cloud AI 100:** `qualcomm.com/qaic: <count>` — Nautilus has 8 Cloud AI 100 Ultra cards × 4 SoCs = 32 devices; each runs LLMs up to ~25B parameters
- **Specific products:** `nvidia.com/a100`, `nvidia.com/rtxa6000`, etc. — see [GPU pods docs](https://nrp.ai/documentation/userdocs/running/gpu-pods/)

Launch it, exec in, and run `nvidia-smi`:


In [ ]:
kubectl apply -n nrp-training-k8s -f my-yamls/gpu-pod.yaml
kubectl get pods -n nrp-training-k8s


**🖥️ Terminal step** — interactive or long-running: use a JupyterLab terminal (**File → New → Terminal**), not this notebook. Ctrl-C / Ctrl-D to exit.

```bash
kubectl exec -it tutorial-$NRP_USER-gpu-pod -n nrp-training-k8s -- nvidia-smi
```


<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
|   0  NVIDIA A10                     On  |   00000000:06:00.0 Off |                    0 |
+-----------------------------------------+------------------------+----------------------+
</pre>
</details>


**Important — GPUs are scarce shared resources.** Delete the pod as soon as you're done:


In [ ]:
kubectl delete pod tutorial-$NRP_USER-gpu-pod -n nrp-training-k8s


## End of episode — cleanup


In [ ]:
kubectl delete pod test-pod-$NRP_USER            -n nrp-training-k8s --ignore-not-found
kubectl delete -f my-yamls/multicontainer.yaml       -n nrp-training-k8s --ignore-not-found
kubectl delete -f my-yamls/pvc.yaml                  -n nrp-training-k8s --ignore-not-found
kubectl delete -f my-yamls/configmap-secret.yaml     -n nrp-training-k8s --ignore-not-found
kubectl delete -f my-yamls/deployment.yaml           -n nrp-training-k8s --ignore-not-found
kubectl delete -f my-yamls/job.yaml                  -n nrp-training-k8s --ignore-not-found
kubectl delete -f my-yamls/ingress-demo.yaml         -n nrp-training-k8s --ignore-not-found
kubectl delete pod tutorial-$NRP_USER-gpu-pod    -n nrp-training-k8s --ignore-not-found

# what did I leave running?
kubectl get all -n nrp-training-k8s


Then verify: `bash check.sh 2` in the workspace (or the last cell of the notebook).


---

## ✅ Check your work

Verifies the state of your resources on the cluster — rerun any time.


In [ ]:
bash check.sh 2
